# Ai_training-model-4 — O'zbekcha Chat AI

**Faqat o'zbek tilida** ishlovchi chat model trainingi.
- **Model:** ~30M params (embed=512, layers=8, heads=8, ff=2048)
- **Vocab:** 16k SentencePiece BPE
- **Vaqt:** 4-5 soat T4 / L4 GPU
- **Datasetlar:** behbudiy/alpaca-cleaned-uz, translation-instruction-uzbek, risqaliyevds/uzbek_instruct, wiki-uz

**Muhim:** `Runtime → Change runtime type → GPU (T4)` ni tanlang.

## 1. Drive mount va GPU tekshirish

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUT  = '/content/drive/MyDrive/Ai_chat_checkpoints_v4'
DRIVE_DATA = '/content/drive/MyDrive/Ai_chat_data_v4'
os.makedirs(DRIVE_OUT, exist_ok=True)
os.makedirs(DRIVE_DATA, exist_ok=True)
print(f'[OUT]  {DRIVE_OUT}')
print(f'[DATA] {DRIVE_DATA}')
!nvidia-smi | head -10

## 2. Repoyni klon qilish

In [ ]:
%cd /content
!rm -rf Ai_training-model-4
!git clone https://github.com/Muhammadjonmurodullayev/Ai_training-model-4.git
%cd Ai_training-model-4
!pip install -q -r requirements.txt
print('[OK] repo + dependencies ready')

## 3. Datasetni tayyorlash (~10-15 daqiqa)

HuggingFace dan o'zbekcha datasetlarni yuklab oladi va ChatML JSONL ga aylantiradi.
Drive da bo'lsa qayta tayyorlamaydi.

In [ ]:
import os, shutil
TRAIN_F = f'{DRIVE_DATA}/chat_train.jsonl'
VAL_F   = f'{DRIVE_DATA}/chat_val.jsonl'

if os.path.exists(TRAIN_F) and os.path.exists(VAL_F):
    print('[CACHE] Drive dan dataset olinadi')
    !mkdir -p data && cp "$TRAIN_F" data/ && cp "$VAL_F" data/
else:
    !python scripts/prepare_dataset.py \
      --output-dir data \
      --sources alpaca-uz,translation-uz,risqaliyevds,wiki-uz \
      --max-samples 250000 --val-ratio 0.01
    !cp data/chat_train.jsonl "$DRIVE_DATA/"
    !cp data/chat_val.jsonl   "$DRIVE_DATA/"
    print('[OK] dataset Drive ga saqlandi')
!ls -lh data/

## 4. Tokenizer training (~3 daqiqa)

In [ ]:
VOCAB_F = f'{DRIVE_OUT}/chat_vocab.model'

if os.path.exists(VOCAB_F):
    print('[CACHE] Drive dan vocab olinadi')
    !cp "$VOCAB_F" data/
    !cp "$DRIVE_OUT/chat_vocab.vocab" data/ 2>/dev/null || true
else:
    !python scripts/train_tokenizer.py \
      --train data/chat_train.jsonl \
      --val data/chat_val.jsonl \
      --output data/chat_vocab \
      --vocab-size 16000
    !cp data/chat_vocab.model "$DRIVE_OUT/"
    !cp data/chat_vocab.vocab "$DRIVE_OUT/"
    print('[OK] tokenizer Drive ga saqlandi')
!ls -lh data/chat_vocab.*

## 5. Training (~4-5 soat)

**Muhim:** `--output-dir` to'g'ridan-to'g'ri **Drive** ga ishora qiladi.
Har `eval-every` stepda eng yaxshi `chat_best.pt` Drive ga yoziladi.
Sessiya uzilsa ham checkpoint xavfsiz.

In [ ]:
!python scripts/train_chat.py \
  --train data/chat_train.jsonl \
  --val data/chat_val.jsonl \
  --tokenizer data/chat_vocab.model \
  --output-dir "$DRIVE_OUT" \
  --embed-dim 512 --num-heads 8 --num-layers 8 --ff-dim 2048 \
  --max-seq-len 512 \
  --batch-size 16 --grad-accum 4 \
  --epochs 6 --lr 3e-4 --warmup-steps 500 \
  --log-every 50 --eval-every 1000

## 6. Resume (sessiya uzilganda)

Agar Colab uzilib qolsa shu cell'ni 1-2 ni qayta ishga tushirib, keyin shu yerdan davom eting.

In [ ]:
RESUME = f'{DRIVE_OUT}/chat_last.pt'
if os.path.exists(RESUME):
    print(f'[RESUME] {RESUME}')
    !python scripts/train_chat.py \
      --train data/chat_train.jsonl \
      --val data/chat_val.jsonl \
      --tokenizer data/chat_vocab.model \
      --output-dir "$DRIVE_OUT" \
      --resume "$RESUME" \
      --embed-dim 512 --num-heads 8 --num-layers 8 --ff-dim 2048 \
      --max-seq-len 512 --batch-size 16 --grad-accum 4 \
      --epochs 6 --lr 3e-4 --warmup-steps 500 \
      --log-every 50 --eval-every 1000
else:
    print('[INFO] resume checkpoint yo\'q, 5-cell ni ishlating')

## 7. Yakuniy ZIP yaratish

In [ ]:
!ls -lh "$DRIVE_OUT/"
!cd "$DRIVE_OUT" && zip -j checkpoints_v4.zip chat_best.pt chat_last.pt chat_vocab.model chat_vocab.vocab 2>/dev/null
!ls -lh "$DRIVE_OUT/checkpoints_v4.zip"
print(f'\n[OK] Drive: {DRIVE_OUT}/checkpoints_v4.zip')
print('Brauzerdan yuklab oling va Ai_chat/backend/incoming/ ga tashlang.')